# 02 — Generate Model Outputs

This notebook demonstrates the generation module. The implementation is in `src/generation.py`; this notebook loads the prompt dataset, runs the two-model generation pipeline, and validates the output.

**Note:** This is the heaviest step because it runs 400 generations on CPU.

In [ ]:
from pathlib import Path
import sys

CURRENT_DIR = Path.cwd()
PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
sys.path.insert(0, str(PROJECT_DIR))

print("Project directory:", PROJECT_DIR)

In [ ]:
import pandas as pd

from src.config import (
    BASE_MODEL,
    INSTRUCT_MODEL,
    MODEL_FAMILY,
    RAW_OUTPUTS_FILENAME,
    PROMPTS_FILENAME,
    get_data_dir,
)
from src.generation import (
    GenerationSettings,
    run_generation_pipeline,
    validate_prompts_for_generation,
    validate_raw_outputs,
)

DATA_DIR = get_data_dir(PROJECT_DIR)
PROMPTS_PATH = DATA_DIR / PROMPTS_FILENAME
RAW_OUTPUTS_PATH = DATA_DIR / RAW_OUTPUTS_FILENAME

print("Base model:", BASE_MODEL)
print("Instruction-tuned model:", INSTRUCT_MODEL)
print("Model family:", MODEL_FAMILY)
print("Prompts path:", PROMPTS_PATH)
print("Raw outputs path:", RAW_OUTPUTS_PATH)

## Load prompts

In [ ]:
prompts_df = pd.read_csv(PROMPTS_PATH)
validate_prompts_for_generation(prompts_df)

print("Loaded prompts:", prompts_df.shape)
display(prompts_df.head(3))

## Generation settings

Generation is deterministic: sampling is disabled, temperature is 0.0, and the random seed is fixed.

In [ ]:
settings = GenerationSettings()
print(settings)

## Run generation

Both models receive the exact same raw prompt text. The expected output is 400 rows: 200 prompts × 2 model types.

In [ ]:
raw_outputs = run_generation_pipeline(
    prompts=prompts_df,
    output_path=RAW_OUTPUTS_PATH,
    settings=settings,
)

print("Raw output shape:", raw_outputs.shape)
display(raw_outputs.head())

## Validate saved outputs

In [ ]:
raw_outputs = pd.read_csv(RAW_OUTPUTS_PATH)
validate_raw_outputs(raw_outputs, expected_prompts=200)

print("Generated output validation passed.")
print("Shape:", raw_outputs.shape)
display(raw_outputs.head())